# Viktor prompt-complexity pipeline

This notebook builds and audits a leakage-safe training dataset from the proprietary Viktor challenge export. It does **not** embed raw data or generated outputs in the notebook.

The generated target is a transparent weak-supervision score, not measured model quality. The raw export has no final model output, usage, timing, or quality label.

## What the builder guarantees

- Every JSONL request becomes one ML example.
- Exact prefix-linked requests are reconstructed into chains.
- Exact, prefix-linked, and semantic near-duplicate task openings stay in one split.
- Normalization is fitted on training rows only; final scores retain their natural weighted distribution.
- Inputs contain only system/developer text, user text, and deterministic text features.
- Logged model and observed execution are target/audit metadata, never predictor inputs.
- Train, validation, and test files are aligned by `request_id`.

In [ ]:
from pathlib import Path
import json
import statistics
import subprocess
import sys
from collections import Counter, defaultdict

ROOT = Path.cwd()
if not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent

EXPORT_DIR = ROOT / 'export'
OUTPUT_DIR = ROOT / 'results' / 'complexity_dataset'
SEED = 42
ROOT, EXPORT_DIR, OUTPUT_DIR

## 1. Build features, targets, metadata, and splits

The split is deterministic. Re-running with the same seed produces the same assignment. Near-duplicate prompt clusters are assigned atomically.

In [ ]:
command = [
    sys.executable,
    str(ROOT / 'scripts' / 'build_complexity_dataset.py'),
    str(EXPORT_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--seed', str(SEED),
    '--train-ratio', '0.70',
    '--validation-ratio', '0.15',
    '--test-ratio', '0.15',
]
subprocess.run(command, cwd=ROOT, check=True)

In [ ]:
manifest = json.loads((OUTPUT_DIR / 'manifest.json').read_text())
summary_keys = [
    'request_count', 'trajectory_count', 'split_group_count',
    'split_request_counts', 'complexity_bands_by_split',
    'label_confidence_by_split', 'split_safety',
]
{key: manifest[key] for key in summary_keys}

## 2. Verify alignment and leakage invariants

These assertions deliberately fail if a request is duplicated, input/target order differs, or a prompt/trajectory appears in more than one split.

In [ ]:
def read_jsonl(path):
    with Path(path).open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

datasets = {}
group_splits = defaultdict(set)
trajectory_splits = defaultdict(set)
all_request_ids = set()

for split in ('train', 'validation', 'test'):
    inputs = read_jsonl(OUTPUT_DIR / f'{split}_inputs.jsonl')
    targets = read_jsonl(OUTPUT_DIR / f'{split}_targets.jsonl')
    metadata = read_jsonl(OUTPUT_DIR / f'{split}_metadata.jsonl')
    input_ids = [row['request_id'] for row in inputs]
    target_ids = [row['request_id'] for row in targets]
    metadata_ids = [row['request_id'] for row in metadata]
    assert input_ids == target_ids == metadata_ids
    assert not (all_request_ids & set(input_ids))
    all_request_ids.update(input_ids)
    for row in metadata:
        group_splits[row['split_group_id']].add(split)
        trajectory_splits[row['trajectory_id']].add(split)
    datasets[split] = (inputs, targets, metadata)

assert len(all_request_ids) == manifest['request_count']
assert all(len(splits) == 1 for splits in group_splits.values())
assert all(len(splits) == 1 for splits in trajectory_splits.values())
print('Verified: aligned files, unique requests, and zero group/trajectory leakage.')

## 3. Inspect score and confidence distributions

Only scores and audit metadata are displayed here; prompts are intentionally not printed.

In [ ]:
for split, (_, targets, _) in datasets.items():
    scores = [row['complexity_score'] for row in targets]
    print({
        'split': split,
        'rows': len(scores),
        'score_min': round(min(scores), 2),
        'score_mean': round(sum(scores) / len(scores), 2),
        'score_max': round(max(scores), 2),
        'bands': dict(Counter(row['complexity_band'] for row in targets)),
        'confidence': dict(Counter(row['label_confidence'] for row in targets)),
    })

In [ ]:
# Detect provider-schema extraction failures and inspect remaining family effects.
model_scores = defaultdict(list)
nonempty_zero_intrinsic = []
for inputs, targets, metadata in datasets.values():
    for model_input, target, audit in zip(inputs, targets, metadata):
        model_scores[audit['logged_model']].append(target['complexity_score'])
        if model_input['user_prompt'].strip() and target['intrinsic_complexity'] == 0:
            nonempty_zero_intrinsic.append(target['request_id'])
assert not nonempty_zero_intrinsic
for model, scores in sorted(model_scores.items(), key=lambda item: -len(item[1])):
    print({
        'model': model, 'rows': len(scores),
        'mean_score': round(statistics.mean(scores), 2),
        'median_score': round(statistics.median(scores), 2),
    })

In [ ]:
# Optional visualization; the data pipeline itself needs no third-party packages.
try:
    import matplotlib.pyplot as plt
except ImportError:
    print('Install matplotlib to display the histogram.')
else:
    plt.figure(figsize=(8, 4))
    for split, (_, targets, _) in datasets.items():
        plt.hist(
            [row['complexity_score'] for row in targets],
            bins=40, alpha=0.5, label=split,
        )
    plt.xlabel('Weak-supervision complexity score')
    plt.ylabel('Requests')
    plt.legend()
    plt.tight_layout()
    plt.show()

## 4. Optional text-model baseline

This cell trains a simple TF–IDF ridge regressor on **user text only**. Validation is used during development; keep the test result for the final evaluation. A production router should also report downstream cost–quality performance, not only regression error.

In [ ]:
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import Ridge
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    from sklearn.pipeline import make_pipeline
except ImportError:
    print('Install scikit-learn to run the optional model baseline.')
else:
    train_inputs, train_targets, _ = datasets['train']
    val_inputs, val_targets, _ = datasets['validation']
    test_inputs, test_targets, _ = datasets['test']

    X_train = [row['user_prompt'] for row in train_inputs]
    y_train = [row['complexity_score'] for row in train_targets]
    X_val = [row['user_prompt'] for row in val_inputs]
    y_val = [row['complexity_score'] for row in val_targets]
    X_test = [row['user_prompt'] for row in test_inputs]
    y_test = [row['complexity_score'] for row in test_targets]

    model = make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=30_000),
        Ridge(alpha=5.0),
    )
    model.fit(X_train, y_train)

    def evaluate(name, X, y):
        prediction = model.predict(X)
        return {
            'split': name,
            'MAE': round(mean_absolute_error(y, prediction), 3),
            'RMSE': round(mean_squared_error(y, prediction) ** 0.5, 3),
            'R2': round(r2_score(y, prediction), 3),
        }

    print(evaluate('validation', X_val, y_val))
    print(evaluate('test', X_test, y_test))

## Files produced

For each split, `*_inputs.jsonl` contains predictor-safe text/features, `*_targets.jsonl` contains the score and its components, and `*_metadata.jsonl` contains audit-only identifiers and the logged model. `all.jsonl` combines them for analysis, while `manifest.json` records formulas, scaling caps, split counts, and limitations.

Before using this score as a router target, manually review a stratified sample from the low/medium/high bands. The score is deliberately auditable, but its weights still encode assumptions that should be validated against human or judge-model ratings.